# POSE — AI-Powered Privacy Policy Intelligence

**Structured LLM extraction + deterministic personalization + Gradio interface**

This notebook is the cleaned public implementation of the POSE competition
prototype.

The pipeline is intentionally hybrid:

```text
privacy-policy text
        ↓
Gemini structured extraction
        ↓
Pydantic validation
        ↓
profile-specific deterministic scoring
        ↓
compact Privacy Snapshot
        ↓
Gradio interface
```

The prototype is an information-assistance tool. It is **not** a legal
compliance checker and the Privacy Fit score is **not** a GDPR score.

The browser-extension architecture shown in the project presentation is a
future product layer; this notebook implements the analysis and personalization
core.

## 1. Dependencies

Install the required packages in your environment if needed:

```bash
pip install google-genai pydantic gradio
```

API credentials are read from the environment rather than embedded in the
notebook:

```bash
export GEMINI_API_KEY="..."
```

Optionally, set a different compatible model:

```bash
export GEMINI_MODEL="..."
```

In [ ]:
import html
import os
from typing import List

import gradio as gr
from google import genai
from pydantic import BaseModel, Field

## 2. Configuration

The public notebook avoids Colab-specific secret handling and hard-coded API
keys.

`GEMINI_API_KEY` is required only when the LLM analysis is actually executed.

In [ ]:
DEFAULT_MODEL = "gemini-3.5-flash-lite"

GEMINI_MODEL = os.getenv(
    "GEMINI_MODEL",
    DEFAULT_MODEL,
)


def create_gemini_client():
    api_key = os.getenv("GEMINI_API_KEY")

    if not api_key:
        raise RuntimeError(
            "GEMINI_API_KEY is not configured. "
            "Set it as an environment variable before running an analysis."
        )

    return genai.Client(api_key=api_key)

## 3. Typed Privacy Schema

POSE does not ask the LLM for an unrestricted summary.

The response is constrained to a typed schema so the downstream product logic
can operate on predictable fields.

In [ ]:
class DataPractice(BaseModel):
    data_type: str = Field(
        description="Type of personal data collected or processed."
    )

    purpose: List[str] = Field(
        description=(
            "All purposes for which this data is collected or processed."
        )
    )

    retention: str = Field(
        description=(
            "Retention period. Use 'Not specified' when absent."
        )
    )

    mandatory_or_optional: str = Field(
        description=(
            "Whether providing the data is mandatory, optional, "
            "or not specified."
        )
    )

    shared_with: List[str] = Field(
        description=(
            "Third parties or categories of recipients. "
            "Use ['Not specified'] when absent."
        )
    )

    evidence: List[str] = Field(
        description=(
            "Short excerpts from the supplied policy supporting "
            "the extraction."
        )
    )


class PrivacyAnalysis(BaseModel):
    company_or_service: str = Field(
        description=(
            "Company, website or service described by the policy. "
            "Use 'Not specified' if unclear."
        )
    )

    data_practices: List[DataPractice]

    user_rights: List[str] = Field(
        description=(
            "Actual data-subject rights explicitly mentioned in the policy."
        )
    )

    attention_points: List[str] = Field(
        description=(
            "Potentially privacy-sensitive or unclear aspects. "
            "Do not claim legal violations."
        )
    )

    simple_summary: str = Field(
        description=(
            "Plain-language summary understandable by a non-lawyer."
        )
    )

## 4. Evidence-Aware Structured Extraction

The prompt encodes several safeguards:

- use only information present in the supplied document;
- do not invent retention periods, recipients or legal status;
- distinguish real privacy rights from cookie-banner buttons;
- avoid unsupported claims of illegality;
- include evidence snippets from the source text;
- distinguish core-service purposes from analytics, profiling and advertising.

In [ ]:
SYSTEM_INSTRUCTIONS = """
You are an AI Privacy Policy Assistant.

Transform complex privacy notices into clear, structured information for
ordinary users.

Analyse ONLY information explicitly contained in the text provided.

For every type of personal data identified, extract:

1. What personal data are collected or processed.
2. ALL purposes for which that data is processed.
3. How long the data are retained.
4. Whether providing the data is mandatory or optional.
5. ALL third parties or categories of third parties receiving the data.
6. Short evidence from the policy supporting the extraction.

Also identify:

7. The company, website or service concerned.
8. Only actual data-subject/privacy rights explicitly mentioned, such as:
   - access
   - deletion/erasure
   - correction/rectification
   - objection
   - restriction
   - data portability
   - withdrawal of consent

Do NOT classify cookie-banner actions such as "Accept all", "Reject all",
"More options", "Manage settings" or similar interface actions as
data-subject rights.

9. Important aspects deserving the user's attention.
10. A simple overall summary.

STRICT RULES:

- Never invent information.
- Analyse only the text provided.
- If retention is absent, write "Not specified".
- If mandatory/optional status is unclear, write "Not specified".
- If recipients are absent, use ["Not specified"].
- If the company/service cannot be determined, write "Not specified".
- Evidence must come directly from the provided privacy policy.
- Do NOT claim that something is illegal or violates GDPR.
- Instead identify unclear, potentially privacy-sensitive or noteworthy
  practices.
- Use simple language understandable by someone without legal knowledge.
- Distinguish core-service purposes from secondary purposes such as
  advertising, marketing, profiling, personalisation and analytics.
""".strip()


def analyse_privacy_policy(
    policy_text: str,
    client=None,
    model: str = GEMINI_MODEL,
) -> PrivacyAnalysis:
    if not policy_text or len(policy_text.strip()) < 50:
        raise ValueError(
            "The supplied privacy-policy text is too short to analyse."
        )

    if client is None:
        client = create_gemini_client()

    prompt = (
        SYSTEM_INSTRUCTIONS
        + "\n\nPRIVACY POLICY:\n\n"
        + policy_text.strip()
    )

    interaction = client.interactions.create(
        model=model,
        input=prompt,
        response_format={
            "type": "text",
            "mime_type": "application/json",
            "schema": PrivacyAnalysis.model_json_schema(),
        },
    )

    return PrivacyAnalysis.model_validate_json(
        interaction.output_text
    )

## 5. Privacy Preference Profiles

The LLM is responsible for extracting facts from the policy.

The preference score is calculated separately through explicit deterministic
weights.

That separation keeps the product logic inspectable and prevents the language
model from inventing a score directly.

In [ ]:
PRIVACY_PROFILES = {
    "🔒 Minimal": {
        "description": "I want to share only what is strictly necessary.",
        "advertising": 20,
        "profiling": 20,
        "analytics": 10,
        "third_party": 15,
        "location": 15,
        "biometric": 25,
        "unspecified_retention": 10,
    },
    "⚖️ Balanced": {
        "description": (
            "I accept reasonable data use but want transparency and control."
        ),
        "advertising": 12,
        "profiling": 15,
        "analytics": 5,
        "third_party": 10,
        "location": 8,
        "biometric": 20,
        "unspecified_retention": 8,
    },
    "✨ Convenience": {
        "description": (
            "I accept additional data processing when it improves the service."
        ),
        "advertising": 5,
        "profiling": 8,
        "analytics": 2,
        "third_party": 5,
        "location": 4,
        "biometric": 15,
        "unspecified_retention": 5,
    },
}

## 6. Deterministic Privacy Fit

The score starts at `100` and subtracts profile-dependent penalties when the
structured extraction contains selected practices.

This is deliberately a **preference-fit heuristic**, not a legal or calibrated
risk metric.

In [ ]:
def calculate_privacy_fit(
    analysis: PrivacyAnalysis,
    profile_name: str,
):
    profile = PRIVACY_PROFILES[profile_name]

    penalty = 0
    reasons = []

    for practice in analysis.data_practices:
        data_type = practice.data_type.lower()

        purposes_text = " ".join(
            practice.purpose
        ).lower()

        if (
            "advertis" in purposes_text
            or "marketing" in purposes_text
        ):
            penalty += profile["advertising"]
            reasons.append(
                f"{practice.data_type} may be used for advertising or marketing."
            )

        if "profil" in purposes_text:
            penalty += profile["profiling"]
            reasons.append(
                f"{practice.data_type} may be used for profiling."
            )

        if (
            "analytics" in purposes_text
            or "analysis" in purposes_text
            or "statistics" in purposes_text
        ):
            penalty += profile["analytics"]
            reasons.append(
                f"{practice.data_type} may be used for analytics."
            )

        if (
            "location" in data_type
            or "gps" in data_type
            or "geolocation" in data_type
        ):
            penalty += profile["location"]
            reasons.append(
                "Location information is collected."
            )

        if (
            "biometric" in data_type
            or "face" in data_type
            or "facial" in data_type
            or "fingerprint" in data_type
            or "voiceprint" in data_type
        ):
            penalty += profile["biometric"]
            reasons.append(
                "Biometric or highly sensitive identifying information "
                "is processed."
            )

        if (
            practice.retention.strip().lower()
            == "not specified"
        ):
            penalty += profile["unspecified_retention"]
            reasons.append(
                f"Retention period for {practice.data_type} is not specified."
            )

        meaningful_recipients = [
            recipient
            for recipient in practice.shared_with
            if recipient.strip().lower() != "not specified"
        ]

        if meaningful_recipients:
            penalty += profile["third_party"]
            reasons.append(
                f"{practice.data_type} may be shared with third parties."
            )

    score = max(0, 100 - penalty)

    # Preserve order while removing duplicate warnings.
    reasons = list(dict.fromkeys(reasons))

    return score, reasons

## 7. Compact User Report

The report intentionally shows only the most decision-relevant information:

- service;
- fit score;
- key data types;
- third-party sharing;
- retention;
- one main warning;
- stated user rights.

This is a product-design choice: the goal is to reduce cognitive load rather
than reproduce the full structured extraction.

In [ ]:
BANNED_RIGHT_ACTIONS = {
    "accept all",
    "reject all",
    "more options",
    "manage settings",
    "privacy settings",
}


def score_label(score: int):
    if score >= 75:
        return "Good match", "🟢"

    if score >= 50:
        return "Some attention needed", "🟡"

    return "Low privacy match", "🔴"


def create_user_report(
    policy_text: str,
    privacy_profile: str,
):
    if not policy_text or len(policy_text.strip()) < 50:
        return (
            '<div class="privacy-card">'
            "⚠️ Please paste a valid privacy policy."
            "</div>"
        )

    try:
        analysis = analyse_privacy_policy(
            policy_text
        )

        score, reasons = calculate_privacy_fit(
            analysis,
            privacy_profile,
        )

        data_types = list(
            dict.fromkeys(
                practice.data_type.strip()
                for practice in analysis.data_practices
                if practice.data_type.strip()
            )
        )

        if len(data_types) > 6:
            data_text = (
                ", ".join(data_types[:6])
                + f" + {len(data_types) - 6} more"
            )
        elif data_types:
            data_text = ", ".join(data_types)
        else:
            data_text = "Not specified"

        third_parties = []

        for practice in analysis.data_practices:
            for recipient in practice.shared_with:
                recipient = recipient.strip()

                if (
                    recipient
                    and recipient.lower() != "not specified"
                    and recipient not in third_parties
                ):
                    third_parties.append(recipient)

        if len(third_parties) > 3:
            third_party_text = (
                ", ".join(third_parties[:3])
                + f" + {len(third_parties) - 3} more"
            )
        elif third_parties:
            third_party_text = ", ".join(third_parties)
        else:
            third_party_text = "Not specified"

        retentions = []

        for practice in analysis.data_practices:
            retention = practice.retention.strip()

            if (
                retention
                and retention.lower() != "not specified"
            ):
                item = (
                    f"{practice.data_type}: {retention}"
                )

                if item not in retentions:
                    retentions.append(item)

        if retentions:
            retention_text = "<br>".join(
                html.escape(item)
                for item in retentions[:3]
            )

            if len(retentions) > 3:
                retention_text += (
                    f"<br>+ {len(retentions) - 3} more"
                )
        else:
            retention_text = "Not specified"

        rights = []

        for right in analysis.user_rights:
            right_lower = right.lower()

            if not any(
                action in right_lower
                for action in BANNED_RIGHT_ACTIONS
            ):
                if right not in rights:
                    rights.append(right)

        if len(rights) > 5:
            rights_text = (
                " · ".join(rights[:5])
                + f" · +{len(rights) - 5} more"
            )
        elif rights:
            rights_text = " · ".join(rights)
        else:
            rights_text = (
                "Not specified in the provided policy"
            )

        if reasons:
            main_warning = reasons[0]
        elif analysis.attention_points:
            main_warning = (
                analysis.attention_points[0]
            )
        else:
            main_warning = (
                "No major points of attention detected "
                "for the selected profile."
            )

        fit_label, icon = score_label(score)

        service = html.escape(
            analysis.company_or_service
        )
        data_text = html.escape(data_text)
        third_party_text = html.escape(
            third_party_text
        )
        rights_text = html.escape(
            rights_text
        )
        main_warning = html.escape(
            main_warning
        )

        return f'''
        <div class="privacy-card">

            <div class="privacy-header">
                <div>
                    <div class="small-label">
                        PRIVACY SNAPSHOT
                    </div>
                    <h2>🔐 {service}</h2>
                </div>

                <div class="score-box">
                    {icon} <b>{score}/100</b>
                    <div class="score-label">
                        {fit_label}
                    </div>
                </div>
            </div>

            <div class="privacy-row">
                <span class="row-icon">👤</span>
                <div>
                    <b>Data collected</b><br>
                    {data_text}
                </div>
            </div>

            <div class="privacy-row">
                <span class="row-icon">🤝</span>
                <div>
                    <b>Shared with</b><br>
                    {third_party_text}
                </div>
            </div>

            <div class="privacy-row">
                <span class="row-icon">⏱️</span>
                <div>
                    <b>Retention</b><br>
                    {retention_text}
                </div>
            </div>

            <div class="warning-box">
                ⚠️ <b>Pay attention</b><br>
                {main_warning}
            </div>

            <div class="rights-box">
                <b>🛡️ Rights explicitly stated</b><br>
                {rights_text}
            </div>

        </div>
        '''

    except Exception as exc:
        # Log the technical error locally, but avoid exposing credentials,
        # stack traces or implementation details in the user-facing UI.
        print(
            "POSE analysis error:",
            repr(exc),
        )

        return (
            '<div class="privacy-card">'
            "<b>❌ Analysis failed</b><br><br>"
            "The policy could not be analysed. "
            "Check the API configuration and try again."
            "</div>"
        )

## 8. Gradio Interface

In [ ]:
CUSTOM_CSS = """
.privacy-card {
    max-width: 680px;
    margin: 25px auto;
    padding: 26px;
    border-radius: 22px;
    background: #18181b;
    border: 1px solid #333;
    box-shadow: 0 12px 35px rgba(0,0,0,0.25);
}

.privacy-header {
    display: flex;
    justify-content: space-between;
    align-items: center;
    gap: 20px;
    margin-bottom: 18px;
}

.privacy-header h2 {
    margin: 4px 0 0 0;
    font-size: 25px;
}

.small-label {
    font-size: 11px;
    opacity: 0.6;
    letter-spacing: 1.5px;
}

.score-box {
    text-align: right;
    font-size: 22px;
    min-width: 130px;
}

.score-label {
    font-size: 12px;
    opacity: 0.7;
    margin-top: 3px;
}

.privacy-row {
    display: flex;
    gap: 14px;
    padding: 14px 0;
    border-top: 1px solid #333;
    line-height: 1.45;
}

.row-icon {
    font-size: 21px;
}

.warning-box {
    margin-top: 14px;
    padding: 14px 16px;
    border-radius: 12px;
    background: rgba(255,190,0,0.10);
    line-height: 1.5;
}

.rights-box {
    margin-top: 14px;
    padding-top: 14px;
    border-top: 1px solid #333;
    line-height: 1.6;
}
"""

In [ ]:
def build_demo():
    with gr.Blocks(
        title="POSE — Privacy Snapshot",
        css=CUSTOM_CSS,
    ) as demo:

        gr.Markdown(
            '''
            # 🔐 POSE — Privacy Snapshot

            ### Understand before you accept.

            Paste a privacy policy and receive a structured,
            profile-aware summary.
            '''
        )

        with gr.Row():
            with gr.Column(scale=2):
                policy_input = gr.Textbox(
                    label="📄 Privacy policy",
                    placeholder=(
                        "Paste the privacy policy here..."
                    ),
                    lines=14,
                )

            with gr.Column(scale=1):
                profile_input = gr.Dropdown(
                    choices=list(
                        PRIVACY_PROFILES.keys()
                    ),
                    value="⚖️ Balanced",
                    label="Your privacy profile",
                )

                analyze_button = gr.Button(
                    "🔍 Analyse before I accept",
                    variant="primary",
                )

        output = gr.HTML()

        analyze_button.click(
            fn=create_user_report,
            inputs=[
                policy_input,
                profile_input,
            ],
            outputs=output,
        )

    return demo


demo = build_demo()

## 9. Launch the Prototype

For local execution:

```python
demo.launch()
```

For notebook environments that require a temporary public link, `share=True`
can be enabled explicitly.

It is disabled by default in the public portfolio version.

In [ ]:
# Uncomment to launch locally.
# demo.launch()

## 10. Limitations and Production Gaps

This prototype intentionally stops short of making production claims.

Current limitations include:

- manual policy input;
- no deployed browser extension;
- no automatic policy crawler;
- no benchmark against expert-labelled extraction;
- no calibrated Privacy Fit weights;
- no formal legal verification;
- potential LLM extraction errors;
- no policy-version tracking;
- no production backend, authentication or persistence.

A production version would need independent evaluation, stronger provenance,
source-linked citations, uncertainty handling and privacy-preserving backend
design.